In [ ]:
import sys
import wandb
import gymnasium as gym
import yaml
import matplotlib.pyplot as plt
import numpy as np
import os

sys.path.append("..\\agents")
sys.path.append("..\\env")
from agents.baselines import RandomAgent, TurnoffDischargeAgent, TurnoffIdleAgent, TurnoffChargeAgent, NeutralDischargeAgent, NeutralIdleAgent, NeutralChargeAgent, TurnonDischargeAgent, TurnonIdleAgent, TurnonChargeAgent, GreedyAgent
from agents.RLAgents import RLAgent
from env.env_microgrid import MicroGridEnv


def load_config(config_file):
    with open(config_file, "r") as f:
        config = yaml.safe_load(f)
    return config

# Load configuration
config = load_config(os.path.join("configs", "config.yaml"))

config['environment']['microgrid']['device']['init_params']['demand']['device']['init_params']['normalisation_factor'] = 2

run = wandb.init(
    project="Test-CommanDER-RL",
    config=config,
    sync_tensorboard=True,
    mode="disabled")


# List all possible baseline_agents
baseline_agents = {
    'random': RandomAgent,
    'turn_off_discharge': TurnoffDischargeAgent,
    'turn_off_idle': TurnoffIdleAgent,
    'turn_off_charge': TurnoffChargeAgent,
    'neutral_discharge': NeutralDischargeAgent,
    'neutral_idle': NeutralIdleAgent,
    'neutral_charge': NeutralChargeAgent,
    'turn_on_discharge': TurnonDischargeAgent,
    'turn_on_idle': TurnonIdleAgent,
    'turn_on_charge': TurnonChargeAgent,
    'greedy': GreedyAgent
}



Decide the agent

In [ ]:
load_model = 'greedy'

In [ ]:
# Create the Gym environment
config['environment']['return_dict'] = load_model in baseline_agents.keys()     # If the agent is a baseline, return the dict
env = gym.make(config['environment']['env_name'],
                env_params=config['environment'])

env = gym.wrappers.RecordEpisodeStatistics(env)

# Load model or create baseline agent

if load_model in baseline_agents.keys():
    model = baseline_agents[load_model](env)
else:
    raise ValueError("Model not found")

Run a trajectory

In [ ]:
trajectory_length = 100


# Run a sample trajectory
obs, info = env.reset()
terminated, truncated = False, False
i = 0

demand = []
commanded_p_grid = []
commanded_status_change = []
implemented_p_grid = []
implemented_status_change = []
implemented_wind_power_setpoint = []
implemented_genset_group_power_setpoint = []
wind_power = []
available_wind_power = []
genset_group_power = []
genset_group_available_power = []
genset_0_power = []
genset_1_power = []
fuel_consumption = []
battery_p_grid = []
battery_soc = []

while not terminated and not truncated and i < trajectory_length:
    # Take action using the model
    action, agent_info = model.predict(obs)
    # Sync observations and actions to WandB

    # Step the environment
    obs, reward, terminated, truncated, env_info = env.step(action)
    demand.append(obs['demand']['demand'])
    commanded_p_grid.append(obs['microgrid']['action_command']['battery']['p_grid'])
    commanded_status_change.append(obs['microgrid']['action_command']['genset_group']['status_change'])
    implemented_p_grid.append(obs['microgrid']['action_implemented']['battery']['p_grid'])
    implemented_status_change.append(obs['microgrid']['action_implemented']['genset_group']['status_change'])
    implemented_wind_power_setpoint.append(obs['microgrid']['action_implemented']['wind_turbine']['turbine_setpoint'])
    implemented_genset_group_power_setpoint.append(obs['microgrid']['action_implemented']['genset_group']['power_setpoint'])
    wind_power.append(obs['wind_turbine']['wind_power'])
    available_wind_power.append(obs['wind_turbine']['available_wind_power'])
    genset_group_power.append(obs['genset_group']['genset_group_active_power'])
    genset_group_available_power.append(obs['genset_group']['genset_group_available_power'])
    genset_0_power.append(obs['genset_group']['gensets'][0]['active_power'])
    genset_1_power.append(obs['genset_group']['gensets'][1]['active_power'])
    fuel_consumption.append(obs['genset_group']['genset_group_fuel_consumption'])
    battery_p_grid.append(obs['battery']['p_grid'])
    battery_soc.append(obs['battery']['soc'])
    
    i += 1
    if i%5 == 0:
        print(i)



In [ ]:

plt.figure()
plt.plot(demand, label='demand')
plt.plot(battery_p_grid, label='battery_p_grid')
plt.plot(wind_power, label='wind_power')
plt.plot(genset_group_power, label='genset_group_power')
plt.title("Power contributions")
plt.legend()


plt.figure()
plt.plot(available_wind_power[:-1], label='available_wind_power')
plt.plot(implemented_wind_power_setpoint[1:], label='implemented_wind_power_setpoint')
plt.plot(wind_power[:-1], label='wind_power', linestyle='--', color='red')
plt.title("Wind power control")
plt.legend()

plt.figure()
plt.plot(battery_p_grid[:-1], label='battery_p_grid')
plt.plot(commanded_p_grid[1:], label='commanded_p_grid')
plt.plot(implemented_p_grid[1:], label='implemented_p_grid')
plt.title("Battery power control")
plt.legend()

plt.figure()
plt.plot(battery_soc, label='battery_soc')
plt.title("Battery state of charge")
plt.legend()

plt.figure()
plt.plot(genset_group_power[:-1], label='genset_group_power')
plt.plot(genset_group_available_power[:-1], label='genset_group_available_power')
plt.plot(genset_0_power[:-1], label='genset_0_power')
plt.plot(genset_1_power[:-1], label='genset_1_power')
plt.plot(implemented_genset_group_power_setpoint[1:], label='implemented_genset_group_power_setpoint')
plt.title("Genset power control")
plt.legend()

plt.figure()
plt.plot(commanded_status_change[1:], label='commanded_status_change')
plt.plot(implemented_status_change[1:], label='implemented_status_change')
plt.plot(np.array(genset_0_power[:-1])/400, label='genset_0_power')
plt.plot(np.array(genset_1_power[:-1])/400, label='genset_1_power')


plt.figure()
plt.plot(fuel_consumption, label='fuel_consumption')
plt.title("Fuel consumption")
plt.legend()



